In [56]:
from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [57]:
! pip list | grep vivarium

# [software verion + hash . date]

vivarium                                 4.0.3
vivarium_build_utils                     3.2.2
vivarium_cluster_tools                   3.1.6
vivarium-compat                          0.6.1
vivarium-dependencies                    1.1.0
vivarium_gates_mncnh                     34.1.dev8+g5eaecaf6f.d20260616 /mnt/share/homes/tylerdy/vivarium_gates_mncnh
vivarium-gbd-mapping                     6.0.1
vivarium_public_health                   5.0.2
vivarium-risk-distributions              3.1.1
vivarium_testing_utils                   0.5.5


In [58]:
! pip freeze | grep vivarium

vivarium==4.0.3
vivarium-compat==0.6.1
vivarium-dependencies==1.1.0
vivarium-gbd-mapping==6.0.1
vivarium-risk-distributions==3.1.1
vivarium_build_utils==3.2.2
vivarium_cluster_tools==3.1.6
-e git+https://github.com/ihmeuw/vivarium_gates_mncnh.git@b8529324faa31aaec00743419d131f765203a25a#egg=vivarium_gates_mncnh
vivarium_public_health==5.0.2
vivarium_testing_utils==0.5.5


In [59]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import InteractiveContext, Artifact

In [60]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification
from pathlib import Path

from vivarium_gates_mncnh.constants.data_values import (
    SIMULATION_EVENT_NAMES,
    COLUMNS,
    PREGNANCY_OUTCOMES,
    PIPELINES,
)

path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
custom_model_specification = build_model_specification(path)
custom_model_specification.configuration.input_data.input_draw_number = 60

custom_model_specification.configuration.population.population_size = 20_000 * 3

included_components = custom_model_specification.components.vivarium_gates_mncnh.components
included_components

['Hemoglobin()',
 'AnemiaInterventionPropensity()',
 'AgelessPopulation("population.scaling_factor")',
 'Pregnancy()',
 'ResultsStratifier()',
 'ANCAttendance()',
 'Ultrasound()',
 'MaternalDisorder("maternal_obstructed_labor_and_uterine_rupture")',
 'MaternalDisorder("maternal_hemorrhage")',
 'MaternalDisorder("maternal_sepsis_and_other_maternal_infections")',
 'ResidualMaternalDisorders()',
 'AbortionMiscarriageEctopicPregnancy()',
 'MaternalDisordersBurden()',
 'LBWSGRisk()',
 "LBWSGRiskEffect('cause.all_causes.all_cause_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_with_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_without_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk')",
 "PretermBirth('neonatal_preterm_birth_with_r

In [61]:
artifact_path = custom_model_specification.configuration.input_data.artifact_path
art = Artifact(artifact_path)

artifact_path

'/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model37.0/ethiopia.hdf'

In [62]:
draw_num = custom_model_specification.configuration.input_data.input_draw_number
draw = 'draw_' + str(draw_num)
draw

'draw_60'

CHECK: puerperal sepsis hemoglobin effect size in the artifact does not vary by sex, age, or year.

Suite: artifact tests.

Type: precise assert.

In [63]:
sepsis_on_hemoglobin_shift = art.load("cause.maternal_sepsis_and_other_maternal_infections.hemoglobin_shift")[draw]
assert len(sepsis_on_hemoglobin_shift) == 2  # early and late postpartum
sepsis_on_hemoglobin_shift

postpartum_period
early_postpartum   -13.247920
late_postpartum     -3.042343
Name: draw_60, dtype: float64

In [ ]:
def check_hemoglobin_shift(sim: InteractiveContext):
    pop = sim.get_population([COLUMNS.MATERNAL_SEPSIS, COLUMNS.PREGNANCY_OUTCOME])
    # We model maternal sepsis on live and still births only
    live_births = pop.loc[
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.STILLBIRTH_OUTCOME)
    ]
    has_sepsis = live_births[COLUMNS.MATERNAL_SEPSIS]
    sepsis_idx = live_births.index[has_sepsis]
    no_sepsis_idx = live_births.index[~has_sepsis]
    if len(sepsis_idx) == 0:
        return
    # Get hemoglobin exposure values from the pipeline
    hgb = sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE).loc[live_births.index]
    mean_hgb_sepsis = hgb.loc[sepsis_idx].mean()
    mean_hgb_no_sepsis = hgb.loc[no_sepsis_idx].mean()

    # is step name the next step to run or the last step which was run? i couldnt check the hemoglobin exposure pipline when
    # steps_remaining was 0
    return {'step': sim._clock.step_name, 'mean_hgb_sepsis': mean_hgb_sepsis, 'mean_hgb_no_sepsis': mean_hgb_no_sepsis}
    # Sepsis shift is negative, so sepsis group should have lower hemoglobin
    #observed_diff = mean_hgb_sepsis - mean_hgb_no_sepsis
    #display(f"{sim._clock.step_name}: ({mean_hgb_sepsis} - {mean_hgb_no_sepsis}) = {observed_diff}")

    # TODO check for age groups as well

In [65]:
def run_all_steps(sim: InteractiveContext):
    step_shifts = []
    while sim.get_number_of_steps_remaining() > 0:
        step_shift = check_hemoglobin_shift(sim)
        if step_shift:
            step_shifts.append(step_shift)
        sim.step()
    return pd.DataFrame(step_shifts)

In [66]:
def run_to_step(sim: InteractiveContext, step_name: str):
    while sim._clock.step_name != step_name:
        sim.step()
    return sim

In [67]:
sim = InteractiveContext(custom_model_specification)
step_shifts = run_all_steps(sim)

2026-07-23 09:52:19.500 | INFO     | simulation_4-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model37.0/ethiopia.hdf.
2026-07-23 09:52:19.502 | INFO     | simulation_4-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-07-23 09:52:19.503 | INFO     | simulation_4-artifact_manager:82 - Artifact additional filter terms are None.


2026-07-23 09:52:50.742 | WARNING  | simulation_4-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-23 09:52:50.744 | WARNING  | simulation_4-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-23 09:52:50.913 | WARNING  | simulation_4-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-07-23 09:52:50.914 | WARNING  | simulation_4-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'ensemble_distribution_weights' during setup.
2026-07-23 09:52:50.914 | WARNING  | simulation_4-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure_standard_deviation' during setup.
2026-07-23 09:52:50.915 | WARNING  | simulation_4-lookup_table_manager:85 - 

In [68]:
step_shifts["observed_diff"] = step_shifts["mean_hgb_sepsis"] - step_shifts["mean_hgb_no_sepsis"]
step_shifts

,step,mean_hgb_sepsis,mean_hgb_no_sepsis,observed_diff
0,residual_maternal_disorders,106.976969,120.502504,-13.525535
1,abortion_miscarriage_ectopic_pregnancy,106.976969,120.502504,-13.525535
2,mortality,106.976969,120.502504,-13.525535
3,early_postpartum,106.976969,120.502504,-13.525535
4,late_postpartum,106.976969,120.502504,-13.525535
5,early_neonatal_mortality,93.729049,120.502504,-26.773456
6,late_neonatal_mortality,103.934626,120.502504,-16.567878
7,postpartum_depression,106.976969,120.502504,-13.525535


In [69]:
step_shifts = step_shifts.set_index("step")[["observed_diff"]].diff().iloc[1:]
#to_plot["target"]= 0
#to_plot.loc["early_neonatal_mortality", "target"] = sepsis_on_hemoglobin_shift["early_postpartum"]

step_shifts

,observed_diff
step,
abortion_miscarriage_ectopic_pregnancy,0.000000
mortality,0.000000
early_postpartum,0.000000
late_postpartum,0.000000
early_neonatal_mortality,-13.247920
late_neonatal_mortality,10.205578
postpartum_depression,3.042343


In [70]:
assert np.isclose(step_shifts.loc["early_neonatal_mortality"], sepsis_on_hemoglobin_shift["early_postpartum"])
# TODO change this to be correct step timing, include return to normal (?), and add late_postpartum

In [55]:
sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE)  # we should be able to get this data at the end of the sim

IndexError: list index out of range